<a href="https://colab.research.google.com/github/Alamsyah-WM/Predict-DNA-binding-protein-with-ML-and-DL/blob/main/BIoInformatic_Group1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [5]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers==4.44.2 datasets==2.19.0 scikit-learn==1.4.2 matplotlib==3.8.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 

#Import Library

In [6]:
import os, math, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

#Config esm2_t6_8M

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = {
    "dataset_name": "PDB1063-186",                 # "PDB1063-186" | "UniSwiss"
    "csv_path": "/content/PDB1063-186.csv",        # ganti ke /content/UniSwiss.csv bila perlu
    "sequence_col": "sequence",                    # sesuaikan kolom
    "label_col": "label",                          # 1 binding, 0 non-binding
    "split_col": "split",                          # "train" | "test" kalau tersedia
    "has_split_col": True,
    "model_type": "BiLSTM",                        # "MLP" | "CNN" | "BiLSTM" | "Transformer"
    "esm_model_name": "facebook/esm2_t6_8M",
    "max_len": 1024,
    "batch_size": 32,
    "epochs": 30,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "use_class_weight": True,                      # aktif untuk PDB
    "scheduler": "step",                           # None | "step" | "cosine" | "plateau"
    "step_size": 5, "gamma": 0.5,
    "patience": 5,                                 # early stopping
    "checkpoint_path": "/content/checkpoints/best.pt",
    "plots_dir": "/content/plots",
    # arsitektur
    "dropout": 0.3,
    "mlp_hidden": [256, 64],
    "cnn_channels": 64, "cnn_kernel": 3,
    "rnn_hidden": 128, "rnn_layers": 1, "bidirectional": True,
    "tr_heads": 2, "tr_layers": 2, "tr_ff": 256
}

os.makedirs(os.path.dirname(config["checkpoint_path"]), exist_ok=True)
os.makedirs(config["plots_dir"], exist_ok=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(42)